In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model=model, lr=1, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="armijo")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.3605155944824219
epoch:  1, loss: 0.22812384366989136
epoch:  2, loss: 0.1490950882434845
epoch:  3, loss: 0.10202439874410629
epoch:  4, loss: 0.0742340087890625
epoch:  5, loss: 0.05801407992839813
epoch:  6, loss: 0.04865995794534683
epoch:  7, loss: 0.04332525283098221
epoch:  8, loss: 0.04031207785010338
epoch:  9, loss: 0.03862348571419716
epoch:  10, loss: 0.03768273815512657
epoch:  11, loss: 0.03716056048870087
epoch:  12, loss: 0.036871012300252914
epoch:  13, loss: 0.036710042506456375
epoch:  14, loss: 0.03661985695362091
epoch:  15, loss: 0.03656849265098572
epoch:  16, loss: 0.036538369953632355
epoch:  17, loss: 0.036519840359687805
epoch:  18, loss: 0.036500632762908936
epoch:  19, loss: 0.03647550940513611
epoch:  20, loss: 0.036467064172029495
epoch:  21, loss: 0.03643253073096275
epoch:  22, loss: 0.03641146048903465
epoch:  23, loss: 0.03639303520321846
epoch:  24, loss: 0.03636404499411583
epoch:  25, loss: 0.036359552294015884
epoch:  26, loss: 

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7317162752151489
Test metrics:  R2 = 0.7142502069473267


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model=model, lr=1, c1=1e-4, c2=0.9, tau=0.1, line_search_method="backtrack", line_search_cond="wolfe")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.6509740948677063
epoch:  1, loss: 0.6196144819259644
epoch:  2, loss: 0.5897476077079773
epoch:  3, loss: 0.561306357383728
epoch:  4, loss: 0.5342420339584351
epoch:  5, loss: 0.5085087418556213
epoch:  6, loss: 0.48410120606422424
epoch:  7, loss: 0.46096518635749817
epoch:  8, loss: 0.4390535354614258
epoch:  9, loss: 0.41828906536102295
epoch:  10, loss: 0.39858976006507874
epoch:  11, loss: 0.3798762857913971
epoch:  12, loss: 0.362118661403656
epoch:  13, loss: 0.34525832533836365
epoch:  14, loss: 0.3292395770549774
epoch:  15, loss: 0.3140079975128174
epoch:  16, loss: 0.2995218336582184
epoch:  17, loss: 0.28574588894844055
epoch:  18, loss: 0.2726442515850067
epoch:  19, loss: 0.26017874479293823
epoch:  20, loss: 0.24832604825496674
epoch:  21, loss: 0.23705743253231049
epoch:  22, loss: 0.22634357213974
epoch:  23, loss: 0.2161579132080078
epoch:  24, loss: 0.2064765840768814
epoch:  25, loss: 0.1972770243883133
epoch:  26, loss: 0.1885370910167694
epoch:

In [8]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6839606761932373
Test metrics:  R2 = 0.6666569709777832


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model=model, lr=10, c1=1e-4, c2=0.9, tau=0.1, line_search_method="backtrack", line_search_cond="strong-wolfe")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.23571467399597168
epoch:  1, loss: 0.14801199734210968
epoch:  2, loss: 0.09868089109659195
epoch:  3, loss: 0.0706947073340416
epoch:  4, loss: 0.05509632080793381
epoch:  5, loss: 0.04658205807209015
epoch:  6, loss: 0.04201795533299446
epoch:  7, loss: 0.03960758075118065
epoch:  8, loss: 0.03834924101829529
epoch:  9, loss: 0.03769799321889877
epoch:  10, loss: 0.03736294060945511
epoch:  11, loss: 0.03719116747379303
epoch:  12, loss: 0.037103071808815
epoch:  13, loss: 0.03705770894885063
epoch:  14, loss: 0.0370340533554554
epoch:  15, loss: 0.0370214506983757
epoch:  16, loss: 0.037014491856098175
epoch:  17, loss: 0.03701034560799599
epoch:  18, loss: 0.03700638189911842
epoch:  19, loss: 0.037000179290771484
epoch:  20, loss: 0.03699657693505287
epoch:  21, loss: 0.03699440509080887
epoch:  22, loss: 0.036988742649555206
epoch:  23, loss: 0.03698542341589928
epoch:  24, loss: 0.03698291629552841
epoch:  25, loss: 0.03697764128446579
epoch:  26, loss: 0.0369

In [10]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.0460856556892395
Test metrics:  R2 = -0.009092569351196289


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model=model, lr_init=1, c1=1e-4, c2=0.9, tau=0.1, line_search_method="backtrack", line_search_cond="goldstein")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.6073564291000366
epoch:  1, loss: 0.6073564291000366
epoch:  2, loss: 0.6073564291000366
epoch:  3, loss: 0.6073564291000366
epoch:  4, loss: 0.6073564291000366
epoch:  5, loss: 0.6073564291000366
epoch:  6, loss: 0.6073564291000366
epoch:  7, loss: 0.6073564291000366
epoch:  8, loss: 0.6073564291000366
epoch:  9, loss: 0.6073564291000366
epoch:  10, loss: 0.6073564291000366
epoch:  11, loss: 0.6073564291000366
epoch:  12, loss: 0.6073564291000366
epoch:  13, loss: 0.6073564291000366
epoch:  14, loss: 0.6073564291000366
epoch:  15, loss: 0.6073564291000366
epoch:  16, loss: 0.6073564291000366
epoch:  17, loss: 0.6073564291000366
epoch:  18, loss: 0.6073564291000366
epoch:  19, loss: 0.6073564291000366
epoch:  20, loss: 0.6073564291000366
epoch:  21, loss: 0.6073564291000366
epoch:  22, loss: 0.6073564291000366
epoch:  23, loss: 0.6073564291000366
epoch:  24, loss: 0.6073564291000366
epoch:  25, loss: 0.6073564291000366
epoch:  26, loss: 0.6073564291000366
epoch:  27,

In [12]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -11380.876953125
Test metrics:  R2 = -11113.5048828125
